# v9 - boundary stress test of raw GLORYS, restricted to the Amazon plume

Redo of v8 with two changes:

1. **`JITParticle`, not `ScipyParticle`.** Measured on this dataset: Scipy costs ~1.96 ms per
   particle-step, JIT is I/O-bound and effectively free per particle (50 particles 19.9 s,
   2000 particles 21.0 s for the same 2-day run). v8 could afford 1650 particles for 8 days;
   v9 runs 2500 for 60 days in a fraction of the time.
2. **Only the Amazon mouth and its shelf**, `lon -53..-44`, `lat -2..+7`. v8 seeded the whole
   GLORYS domain, which reaches 95W into the eastern Pacific and 30N into the Caribbean. Those
   losses said nothing about this study.

Same rules as v8: `FieldSet.from_nemo` (which negates W itself), the stock `AdvectionRK4_3D`
imported rather than retyped, and `CheckError`. No `Diagnose`, no reflection, no clamping, no
velocity cap. Every boundary quantity is computed **after the run** from the written trajectory
and the mesh.

## What the region actually contains

Surveyed before choosing the classes - 8442 wet columns:

| depth band | columns |
|---|---|
| `H < 20 m` (river mouth, inner shelf) | 1422 |
| `20-100 m` (shelf) | 1778 |
| `100-1000 m` (slope) | 614 |
| `1000-3000 m` | 1069 |
| `> 3000 m` | 3559 |

230 coastline columns, 6032 with a partial-cell gap over 1 m, and **max depth 4798 m**.

## v8's findings carried in, and what they imply here

- **The bottom depth-axis truncation cannot fire in this region.** v8 found Parcels' depth axis
  ends at 5500.0015 m while the mesh seabed reaches 5958 m, killing every particle placed in the
  8.84% of columns deeper than that. In this box **zero columns exceed 5500 m**, so `floor_face`
  and `deepest_cell` should be clean here. That is prediction 2 below, and it is the reason the
  region matters: a defect that dominated the global test is irrelevant to the Amazon.
- **The surface defect should fire exactly as before.** v8: 69 of 70 particles that died within
  3 h of being seeded at `z = 0` had upward stored `W(0)`, 99.3% agreement, on a displacement of
  0.05 mm per step. Nothing about that is regional.
- **Nothing flew away** - global max implied speed 1.26 m/s. The NBC runs faster than anything
  in v8's sample, so this is the sharper test of that.

## Predictions, recorded before the run

1. `surface` loses most of its particles, and the deaths are predicted by `W(0) > 0` at the seed.
2. `floor_face`, `partial_gap` and `deepest_cell` lose nothing - the seabed is above the axis here.
3. `coast` and `mouth_shallow` are the classes at risk instead: the Amazon shelf is where land
   masking, partial cells and strong shear all coincide.
4. No implied speed above 3 m/s anywhere, despite the NBC.
5. `interior_deep` and `plume_5m` - the realistic release - lose nothing.

Section 8 reads all five back.

*(`seed_rng`, never `rng`: `Kernel.__init__` overwrites `math`, `ParcelsRandom`, `rng`, `random`
and `StatusCode` in the notebook's globals, `parcels/kernel.py:219-225`.)*

## 1. Configuration

In [ ]:
n_per_class        = 250
run_time_days      = 60
time_step_minutes  = 20
out_put_step_hours = 3

ref_date, offset = "1993-01-01", 1
rdm_seed = 110987
MONTHS = [1, 2, 3]

Hgr, Zgr = "Hgr_cmesh.nc", "Zgr_cmesh2.nc"
def mf(pat): return [pat.format(y=1993, m=m) for m in MONTHS]
GLORYS = dict(U=mf("U_{y}-{m:02d}.nc"), V=mf("V_{y}-{m:02d}.nc"), W=mf("W_{y}-{m:02d}.nc"))

# the study region: Amazon mouth, shelf and slope
AMZ   = dict(lon=(-53.0, -44.0), lat=(-2.0, 7.0))
MOUTH = dict(lon=(-51.5, -47.5), lat=(-1.5, 2.5))    # the river mouth proper

SPEED_BINS = [1.0, 2.0, 3.0, 5.0]
out_root    = "tracks_v9"
FORCE_RERUN = False

In [ ]:
import os, sys, shutil, contextlib, textwrap, time, inspect
import numpy as np
import xarray as xr
from datetime import timedelta
from scipy.spatial import cKDTree
import matplotlib as mpl
import matplotlib.pyplot as plt

import parcels
from parcels import (FieldSet, ParticleSet, JITParticle, Variable, AdvectionRK4_3D, StatusCode)

@contextlib.contextmanager
def quiet():
    sys.stdout.flush()
    saved, devnull = os.dup(1), os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(devnull, 1); yield
    finally:
        os.dup2(saved, 1); os.close(devnull); os.close(saved)

print("parcels", parcels.__version__)
miss = [p for v in ("U", "V", "W") for p in GLORYS[v] if not os.path.exists(p)]
print("GLORYS", "OK" if not miss else "MISSING: " + ", ".join(miss))
start_time = np.datetime64(ref_date) + np.timedelta64(offset, "D")
nout = int(run_time_days * 24 / out_put_step_hours) + 1
print(f"{start_time} -> {start_time + np.timedelta64(run_time_days,'D')}"
      f"   dt={time_step_minutes} min   {nout} outputs")
CODES = {getattr(StatusCode, n): n for n in
         ("Success", "Evaluate", "Repeat", "Delete", "StopExecution", "Error",
          "ErrorInterpolation", "ErrorOutOfBounds", "ErrorThroughSurface",
          "ErrorTimeExtrapolation") if hasattr(StatusCode, n)}

## 2. Mesh and the post-hoc lookup

In [ ]:
zgr = xr.open_dataset(Zgr).squeeze()
hgr = xr.open_dataset(Hgr).squeeze()
mbathy  = zgr.mbathy.values.astype(int)
gdepw_0 = zgr.gdepw_0.squeeze().values.astype("f8")
e3t_ps  = zgr.e3t_ps.values.astype("f8")
lon2d, lat2d = zgr.nav_lon.values.astype("f8"), zgr.nav_lat.values.astype("f8")
nz, ny, nx = len(gdepw_0), *mbathy.shape
kb    = np.clip(mbathy - 1, 0, nz - 1)
thick = np.where(mbathy > 0, e3t_ps, 0.0)
H     = gdepw_0[kb] + thick
Gnom  = np.where(mbathy > 0, gdepw_0[np.clip(mbathy, 0, nz - 1)], 0.0)
zbot  = gdepw_0[kb] + 0.5 * thick
ocean = mbathy > 0
AXIS  = float(xr.open_dataset(GLORYS["W"][0]).depthw.values[-1])

tree = cKDTree(np.column_stack([lon2d.ravel(), lat2d.ravel()]))
FLAT = dict(mbathy=mbathy.ravel(), H=H.ravel(), Gnom=Gnom.ravel())
def lookup(lo, la):
    lo, la = np.asarray(lo, float), np.asarray(la, float)
    ok = np.isfinite(lo) & np.isfinite(la)
    out = {k: np.full(lo.shape, np.nan) for k in ("H", "Gnom")}
    land = np.zeros(lo.shape, bool)
    if ok.any():
        _, idx = tree.query(np.column_stack([lo[ok], la[ok]]))
        land[ok] = FLAT["mbathy"][idx] == 0
        for k in ("H", "Gnom"): out[k][ok] = FLAT[k][idx]
    return land, out["H"], out["Gnom"]

inbox = ((lon2d >= AMZ["lon"][0]) & (lon2d <= AMZ["lon"][1])
         & (lat2d >= AMZ["lat"][0]) & (lat2d <= AMZ["lat"][1]))
inmouth = ((lon2d >= MOUTH["lon"][0]) & (lon2d <= MOUTH["lon"][1])
           & (lat2d >= MOUTH["lat"][0]) & (lat2d <= MOUTH["lat"][1]))
wet = ocean & inbox
print(f"Amazon box: {int(wet.sum())} wet columns, H {H[wet].min():.1f}..{H[wet].max():.1f} m")
print(f"columns deeper than the Parcels depth axis ({AXIS:.1f} m): {int((wet & (H > AXIS)).sum())}"
      "   <- v8's bottom defect cannot fire here")

## 3. Ten seed classes, all inside the Amazon box

In [ ]:
seed_rng = np.random.default_rng(rdm_seed)
def pick(mask, n):
    jj, ii = np.where(mask)
    if len(jj) == 0: return np.array([], int), np.array([], int)
    s = seed_rng.choice(len(jj), size=n, replace=len(jj) < n)
    return jj[s], ii[s]

land_nb = np.zeros_like(ocean)
for sh, ax in ((1, 0), (-1, 0), (1, 1), (-1, 1)):
    land_nb |= np.roll(~ocean, sh, axis=ax)

CLASSES = {}
def add(name, mask, zfun):
    j, i = pick(mask, n_per_class)
    if len(j) == 0: print(f"  {name:<14} EMPTY"); return
    CLASSES[name] = (lon2d[j, i], lat2d[j, i], np.asarray(zfun(j, i), float))
mid = lambda j, i: 0.5 * H[j, i]

add("surface",       wet,                                    lambda j, i: np.zeros(len(j)))
add("subsurf_1cm",   wet,                                    lambda j, i: np.full(len(j), 0.01))
add("plume_5m",      wet & inmouth,                          lambda j, i: np.full(len(j), 5.0))
add("floor_face",    wet,                                    lambda j, i: Gnom[j, i])
add("partial_gap",   wet & (Gnom - H > 1.0),
    lambda j, i: H[j, i] + seed_rng.uniform(0.05, 0.95, len(j)) * (Gnom[j, i] - H[j, i]))
add("deepest_cell",  wet,                                    lambda j, i: zbot[j, i])
add("coast",         wet & land_nb,                          mid)
add("mouth_shallow", wet & inmouth & (H < 20),               mid)
add("shelf_break",   wet & (H > 100) & (H < 1000),           mid)
add("interior_deep", wet & (H > 3000),                       lambda j, i: np.full(len(j), 500.0))

NAMES = list(CLASSES)
start_lon = np.concatenate([CLASSES[k][0] for k in NAMES])
start_lat = np.concatenate([CLASSES[k][1] for k in NAMES])
start_dep = np.concatenate([CLASSES[k][2] for k in NAMES])
start_cls = np.concatenate([np.full(len(CLASSES[nm][0]), k) for k, nm in enumerate(NAMES)])
npart = len(start_lon)
print(f"{'class':<15}{'n':>5}{'uniq cols':>11}{'z min':>9}{'z max':>9}{'H min':>9}{'H max':>9}")
for nm in NAMES:
    lo, la, z = CLASSES[nm]
    _, hh, _ = lookup(lo, la)
    print(f"{nm:<15}{len(lo):>5}{len(set(map(tuple, np.c_[lo, la]))):>11}"
          f"{z.min():>9.2f}{z.max():>9.2f}{np.nanmin(hh):>9.1f}{np.nanmax(hh):>9.1f}")
print(f"\n{npart} particles x {run_time_days} days, JITParticle")

## 4. Kernels - stock advection and `CheckError`, nothing else

In [ ]:
def CheckError(particle, fieldset, time):  # pragma: no cover
    '''Records the status code that killed a particle, then deletes it. Writes a status code and
    nothing else - no position, depth or velocity is touched.'''
    if particle.state >= 50:
        particle.died = particle.state
        particle.delete()

class BoundaryParticle(JITParticle):
    cls  = Variable("cls",  initial=0, dtype=np.int32, to_write="once")
    died = Variable("died", initial=0, dtype=np.int32)

print("kernels:", [AdvectionRK4_3D.__name__, CheckError.__name__], "| particle: JITParticle")
print(textwrap.indent("".join(inspect.getsource(AdvectionRK4_3D).splitlines(keepends=True)[:3]), "  "))

## 5. Run

In [ ]:
def run():
    out = os.path.join(out_root, "amazon_boundaries.zarr")
    if os.path.exists(out) and not FORCE_RERUN:
        print("reusing", out); return out
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out_root, exist_ok=True)
    fn = {v: {"data": GLORYS[v], "lon": Hgr, "lat": Hgr, "depth": GLORYS["W"][0]}
          for v in ("U", "V", "W")}
    fs = FieldSet.from_nemo(fn, {"U": "vozocrtx", "V": "vomecrty", "W": "vovecrtz"},
                            {v: {"lon": "glamf", "lat": "gphif", "depth": "depthw",
                                 "time": "time_counter"} for v in ("U", "V", "W")},
                            deferred_load=True, allow_time_extrapolation=False)
    ps = ParticleSet(fs, BoundaryParticle, lon=start_lon, lat=start_lat, depth=start_dep,
                     time=np.datetime64(start_time), cls=start_cls)
    pf = ps.ParticleFile(name=out, outputdt=timedelta(hours=out_put_step_hours),
                         chunks=(npart, nout + 2))
    t0 = time.time()
    with quiet():
        ps.execute([AdvectionRK4_3D, CheckError], runtime=timedelta(days=run_time_days),
                   dt=timedelta(minutes=time_step_minutes), output_file=pf)
    print(f"  {time.time()-t0:.0f} s")
    return out

path = run()
d = xr.open_zarr(path).compute()
LON, LAT, Z = d.lon.values, d.lat.values, d.z.values
CLS  = d.cls.values
DIED = np.nanmax(np.nan_to_num(d.died.values), axis=1).astype(int)
NL   = np.isfinite(LAT).sum(1); full = NL.max(); dead = NL < full
tdays = np.arange(LAT.shape[1]) * out_put_step_hours / 24.0
print(f"{LAT.shape[0]} particles x {LAT.shape[1]} obs; longest-lived {full} -> day {full*out_put_step_hours/24:.1f}")

## 6. Survival, cause of loss, and how far they got first

In [ ]:
R = 6371000.0
la1, la2 = np.deg2rad(LAT[:, :-1]), np.deg2rad(LAT[:, 1:])
lo1, lo2 = np.deg2rad(LON[:, :-1]), np.deg2rad(LON[:, 1:])
hv = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
step_m = 2*R*np.arcsin(np.sqrt(np.clip(hv, 0, 1)))
vh = step_m/(out_put_step_hours*3600.0)
vz = np.abs(np.diff(Z, axis=1))/(out_put_step_hours*3600.0)
path_km = np.nansum(step_m, axis=1)/1000.0
li = np.clip(NL-1, 0, None); ar = np.arange(len(NL))
lastlon, lastlat, lastz = LON[ar, li], LAT[ar, li], Z[ar, li]

ecodes = [c for c in sorted(CODES) if c >= 50]
print(f"{'class':<15}{'n':>4}{'alive':>7}{'lost':>6}" + "".join(
      f"{CODES[c].replace('Error',''):>15}" for c in ecodes)
      + f"{'med days':>10}{'med km':>9}")
for k, nm in enumerate(NAMES):
    s = CLS == k; l = s & dead
    row = f"{nm:<15}{int(s.sum()):>4}{int((s & ~dead).sum()):>7}{int(l.sum()):>6}"
    for c in ecodes:
        n_ = int((l & (DIED == c)).sum()); row += f"{n_ if n_ else '.':>15}"
    row += (f"{np.median(NL[l])*out_put_step_hours/24:>10.2f}{np.median(path_km[l]):>9.1f}"
            if l.any() else f"{'-':>10}{'-':>9}")
    print(row)
print(f"{'TOTAL':<15}{npart:>4}{int((~dead).sum()):>7}{int(dead.sum()):>6}" + "".join(
      f"{int((dead & (DIED == c)).sum()) or '.':>15}" for c in ecodes))

## 7. Boundary excursions and fly-away check, both post-hoc

In [ ]:
land_hit, H_at, G_at = lookup(LON, LAT)
above_S = np.where(np.isfinite(Z), -Z, np.nan)
below_F = np.where(np.isfinite(Z), Z - G_at, np.nan)
below_H = np.where(np.isfinite(Z), Z - H_at, np.nan)
def npart_over(x, s):
    a = np.where(np.isfinite(x[s]), x[s], -np.inf)
    return int((a.max(axis=1) > 1e-6).sum())
def mx(x, s): return np.nanmax(x[s]) if np.isfinite(x[s]).any() else np.nan

print(f"{'class':<15}{'aboveSurf':>10}{'max m':>8}{'belowFloor':>12}{'max m':>8}"
      f"{'belowH':>8}{'max m':>9}{'overLand':>10}")
for k, nm in enumerate(NAMES):
    s = CLS == k
    print(f"{nm:<15}{npart_over(above_S,s):>10}{max(mx(above_S,s),0):>8.3f}"
          f"{npart_over(below_F,s):>12}{max(mx(below_F,s),0):>8.3f}"
          f"{npart_over(below_H,s):>8}{max(mx(below_H,s),0):>9.2f}"
          f"{int(land_hit[s].any(axis=1).sum()):>10}")

print(f"\n{'class':<15}{'max |vh|':>10}{'p99.9':>9}{'median':>9}{'max |vz|':>10}"
      + "".join(f"{'>'+str(b):>7}" for b in SPEED_BINS))
for k, nm in enumerate(NAMES):
    s = CLS == k; a = vh[s]; g = np.isfinite(a)
    if not g.any(): continue
    row = (f"{nm:<15}{np.nanmax(a):>10.3f}{np.nanpercentile(a[g],99.9):>9.3f}"
           f"{np.nanmedian(a[g]):>9.3f}{mx(vz,s):>10.4f}")
    for t in SPEED_BINS:
        n_ = int((np.where(np.isfinite(a), a, -np.inf).max(axis=1) > t).sum())
        row += f"{n_ if n_ else '.':>7}"
    print(row)
print(f"\nglobal max implied horizontal speed {np.nanmax(vh):.3f} m/s   vertical {np.nanmax(vz):.4f} m/s")

## 8. The surface mechanism, re-tested in this region

In [ ]:
w = xr.open_dataset(GLORYS["W"][0]).vovecrtz
w0 = 0.5*(w.isel(time_counter=0, depthw=0).values.astype("f8")
          + w.isel(time_counter=1, depthw=0).values.astype("f8"))
k = NAMES.index("surface"); s = CLS == k
_, ii = tree.query(np.column_stack([LON[s,0], LAT[s,0]])); jj, i2 = np.unravel_index(ii, mbathy.shape)
ws = w0[jj, i2]; dd = dead[s]; firstdeath = (NL[s] <= 1) & dd
print(f"surface class: {int(firstdeath.sum())} died in the first {out_put_step_hours} h, "
      f"{int((dd&~firstdeath).sum())} later, {int((~dd).sum())} survived")
for lab, m in (("died first step", firstdeath), ("died later", dd & ~firstdeath), ("survived", ~dd)):
    if m.sum() == 0: continue
    print(f"  {lab:<16} W(0) upward: {int((ws[m]>0).sum()):>4}/{int(m.sum()):<4}"
          f"  mean {np.nanmean(ws[m]):+.3e} m/s")
print(f"  'dies at once <=> stored W(0) > 0' agreement: {((ws>0)==firstdeath).mean()*100:.1f}%")
dt_s = time_step_minutes*60
print(f"  displacement in one step at median |W(0)|: {np.nanmedian(np.abs(ws))*dt_s*1e3:.4f} mm"
      f"   (max {np.nanmax(np.abs(ws))*dt_s*1e3:.3f} mm)")

## 9. Figures

In [ ]:
SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dedcd6"
plt.rcParams.update({"font.size":9,"axes.edgecolor":GRID,"axes.labelcolor":INK2,
 "xtick.color":INK2,"ytick.color":INK2,"axes.facecolor":SURF,"figure.facecolor":SURF,
 "axes.titlecolor":INK})
cmap = mpl.colormaps["turbo"](np.linspace(0.05, 0.95, len(NAMES)))
CCOL = {61:"#c8442b", 60:"#2a78d6", 51:"#e0a11b", 70:"#1baf7a"}
NT = max(full-1, 1)

fig, axs = plt.subplots(2, 2, figsize=(13, 9.5))
ax = axs[0,0]
for k, nm in enumerate(NAMES):
    s = CLS == k
    ax.plot(tdays[:NT], np.isfinite(LAT[s][:,:NT]).sum(0)/s.sum()*100, lw=1.6, color=cmap[k], label=nm)
ax.set_xlabel("days"); ax.set_ylabel("% still alive"); ax.set_ylim(-2,103)
ax.set_xlim(0, tdays[NT-1])
ax.set_title("a  survival by boundary class", loc="left", fontsize=10.5, pad=8)
ax.legend(frameon=False, fontsize=6.6, ncol=2)

ax = axs[0,1]
bot = np.zeros(len(NAMES))
for c in ecodes:
    v = np.array([int(((CLS==k)&dead&(DIED==c)).sum()) for k in range(len(NAMES))])
    if v.sum()==0: continue
    ax.barh(np.arange(len(NAMES)), v, left=bot, color=CCOL.get(c,"#888"),
            label=CODES[c].replace("Error",""), height=0.7); bot += v
ax.barh(np.arange(len(NAMES)), [int(((CLS==k)&~dead).sum()) for k in range(len(NAMES))],
        left=bot, color="#d8d5cc", label="survived", height=0.7)
ax.set_yticks(np.arange(len(NAMES))); ax.set_yticklabels(NAMES, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("particles"); ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title("b  cause of loss", loc="left", fontsize=10.5, pad=8)

ax = axs[1,0]
x0,x1 = AMZ["lon"]; y0,y1 = AMZ["lat"]
mreg = (lon2d>=x0-3)&(lon2d<=x1+3)&(lat2d>=y0-3)&(lat2d<=y1+3)
gj,gi = np.where(mreg); Js,Is = slice(gj.min(),gj.max()+1), slice(gi.min(),gi.max()+1)
BATH = np.where(ocean[Js,Is], H[Js,Is], np.nan)
LEV=[0,10,20,50,100,200,500,1000,2000,4000]
bc = mpl.colors.ListedColormap(mpl.colormaps["Greys"](np.linspace(0.06,0.55,len(LEV)-1)))
bc.set_bad("#e6e1d6")
ax.set_facecolor("#e6e1d6")
ax.pcolormesh(lon2d[Js,Is], lat2d[Js,Is], BATH, cmap=bc,
              norm=mpl.colors.BoundaryNorm(LEV,bc.N), shading="auto", zorder=0)
ax.add_patch(plt.Rectangle((x0,y0),x1-x0,y1-y0,fill=False,ec="#c8442b",lw=1.3,ls="--",zorder=6))
for c in ecodes:
    m = dead & (DIED==c)
    if not m.any(): continue
    ax.scatter(lastlon[m], lastlat[m], s=15, color=CCOL.get(c,"#888"), edgecolor="none",
               alpha=0.85, zorder=4, label=f"{CODES[c].replace('Error','')} n={int(m.sum())}")
ax.set_xlim(x0-3,x1+3); ax.set_ylim(y0-3,y1+3)
ax.set_aspect(1/np.cos(np.deg2rad((y0+y1)/2)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(frameon=False, fontsize=7.5, loc="upper right")
ax.set_title("c  where lost particles were last written (seed box dashed)", loc="left", fontsize=10.5, pad=8)

ax = axs[1,1]
bins = np.linspace(np.nanmin(ws)*1e6, np.nanmax(ws)*1e6, 34)
ax.hist(ws[firstdeath]*1e6, bins=bins, color="#c8442b", alpha=.85, label=f"died at once (n={firstdeath.sum()})")
ax.hist(ws[dd&~firstdeath]*1e6, bins=bins, color="#e8a598", alpha=.85, label=f"died later (n={(dd&~firstdeath).sum()})")
ax.hist(ws[~dd]*1e6, bins=bins, color="#2a78d6", alpha=.85, label=f"survived (n={(~dd).sum()})")
ax.axvline(0, color=INK, lw=1.1)
ax.set_xlabel("stored W(0) at the seed  (1e-6 m/s)"); ax.set_ylabel("particles")
ax.set_title("d  surface: the sign of W(0) decides the fate", loc="left", fontsize=10.5, pad=8)
ax.legend(frameon=False, fontsize=7.5)
for a in axs.ravel():
    for sp in ("top","right"): a.spines[sp].set_visible(False)
fig.suptitle(f"v9 - raw GLORYS boundary test, Amazon mouth and shelf, {npart} JIT particles, "
             f"{run_time_days} days", x=0.005, ha="left", fontsize=12.5, color=INK, y=0.998)
fig.tight_layout(rect=[0,0,1,0.95]); fig.savefig("v9_boundaries.png", dpi=160, facecolor=SURF)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16.5, 4.6))
ax = axs[0]
for k, nm in enumerate(NAMES):
    a = vh[CLS==k]; a = a[np.isfinite(a)&(a>0)]
    if a.size < 10: continue
    q = np.linspace(0,100,200)
    ax.plot(np.percentile(a,q), 100-q, lw=1.4, color=cmap[k], label=nm)
ax.axvline(2.0, color="#c8442b", lw=1.2, ls="--")
ax.text(2.15, 20, "2 m/s\n(NBC peak)", color="#c8442b", fontsize=7.5)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("implied horizontal speed (m/s)"); ax.set_ylabel("% of steps exceeding")
ax.set_title(f"a  fly-away check - global max {np.nanmax(vh):.2f} m/s", loc="left", fontsize=10)
ax.legend(frameon=False, fontsize=6.4, ncol=2)

ax = axs[1]
for nm in ("surface","subsurf_1cm","plume_5m","interior_deep"):
    if nm not in NAMES: continue
    k = NAMES.index(nm); s = CLS==k
    zz = np.where(np.isfinite(LAT[s]), Z[s], np.nan)[:, :NT]
    ax.plot(tdays[:NT], np.nanmedian(zz, axis=0), lw=1.7, color=cmap[k], label=nm)
ax.invert_yaxis(); ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("days"); ax.set_ylabel("depth (m)")
ax.set_title("b  near-surface classes - median depth", loc="left", fontsize=10)

ax = axs[2]
for nm in ("floor_face","partial_gap","deepest_cell","coast","mouth_shallow","shelf_break"):
    if nm not in NAMES: continue
    k = NAMES.index(nm); s = CLS==k
    zz = np.where(np.isfinite(LAT[s]), Z[s], np.nan)[:, :NT]
    ax.plot(tdays[:NT], np.nanmedian(zz, axis=0), lw=1.7, color=cmap[k], label=nm)
ax.invert_yaxis(); ax.legend(frameon=False, fontsize=8)
ax.set_xlabel("days"); ax.set_ylabel("depth (m)")
ax.set_title("c  seabed and shelf classes - median depth", loc="left", fontsize=10)
for a in axs:
    for sp in ("top","right"): a.spines[sp].set_visible(False)
fig.suptitle("speeds and vertical behaviour", x=0.005, ha="left", fontsize=12.5, color=INK, y=0.995)
fig.tight_layout(rect=[0,0,1,0.93]); fig.savefig("v9_speeds_depth.png", dpi=160, facecolor=SURF)
plt.show()

## 10. Reading the result

Filled in after the run, against the five predictions in the header.